# Bölüm 16 · Çok Düzeyli Modeller

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. İç içe veride birimleri ve bağımlılığı tanımlar.
2. Grup içi ve gruplar arası hedefleri ayırır.
3. Merkezleme ve yeni grup tahminini gerekçelendirir.

**İleri rota:** Rastgele eğimler, sınırda testler, REML ve hedef ağırlıkları; Bölüm 8 ve 12 sonrasında.

**Ç1 · Çıkış sorusu:** ICC=.10, her regresyon katsayısının tasarım etkisinin aynı olduğu anlamına gelir mi?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b16-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b16-cok-duzeyli-modeller'
modules=['uygulamalar/b16-cok-duzeyli-modeller/okul-tahmini']
expected_main=json.loads((root/"bolumler/b16/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 16 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b16/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 16 — görevler

Güncel Revizyon 19 görevleri; önce kendiniz çözün.

## D1–D3: Analizden Önce

1.  Bu veride düzey 1 ve düzey 2 birimleri nelerdir? SES ve okul türü hangi düzeydedir?

2.  7185 öğrenciyi bağımsız sayan bir regresyonda okul türü katsayısının standart hatası neden fazla küçük çıkar?

3.  Okul ortalama SES’ini kaynaktaki sütundan almak yerine örneklemden hesaplamanın gerekçesi ne olabilir?

Yanıtlar Ek F, Bölüm 16’dadır (s. kitaptaki ilgili sayfa).

# Rehberli ve Bağımsız Etkinlikler

## G1–G4: Rehberli Çözüm

1.  Boş model varyanslarından ICC’yi, tasarım etkisini ve etkili örneklem büyüklüğünü hesaplayın.

2.  14 öğrencili bir okulun ortalamasının güvenirliğini ($\lambda_j$) bulun. BLUP bu okulun ortalamasını neden daha çok büzer?

3.  Grup içi ve gruplar arası eğimlerden bağlamsal etkiyi bulun ve bir cümleyle yorumlayın.

4.  Rastgele eğim LR testinin karışım p değerini hesaplayın. Neden $\chi^2_2$ kullanılmaz?

## B1–B3: Bağımsız Çözüm

1.  Düzeyler arası etkileşim modelinden okul ortalama SES’i $+0.5$ olan bir Katolik okulda okul içi SES eğimini hesaplayın.

2.  Eğim varyansındaki %86’lık azalmayı ve bunun neden bir “$R^2$” gibi dikkatle okunması gerektiğini açıklayın.

3.  Naif OLS, küme-sağlam OLS ve rastgele kesişimli modelin sabit terim standart hatalarını karşılaştırın. Hangisi hangi soruya yanıt verir?

## H1: Hatalı Raporu Onarma

Beş iddiayı düzeltin.

## P1: Çok Düzeyli Çalışma Planı

Türkiye’de sınıflar içindeki öğrencilerde bir okuma müdahalesinin etkisini inceleyecek bir çalışma için plan yazın: düzeyler ve atama birimi (öğrenci mi sınıf mı), beklenen ICC ve buna göre sınıf/öğrenci sayısı, merkezleme kararları, rastgele etki yapısı, kestirim (ML/REML), düzey 2 örneklemi ve sd düzeltmesi, raporlanacak tablolar. Rubrik: tasarım 25, örneklem 20, model 30, raporlama 15, yeniden üretim 10 puan.

## T1–T3: Hedef ve Tahmin Denetimi

1.  HSB’deki üç ortalamayı ve ağırlıklarını karşılaştırın. Hangisi ulusal ortalamadır? Okul büyüklüğü sonuçla ilişkiliyse neyi sorgularsınız?

2.  Eşit ve eşitsiz büyüklük tasarım etkilerini karşılaştırın. 757.95 sayısını okul türü katsayısına doğrudan uygulamak neden yanlıştır?

3.  Okul 8367 için okul ortalaması ve yeni öğrenci aralıklarını ayırın. Yeni okuldan 30 öğrenci alınca okul varyansı neden kaybolmaz? Yeni okula genelleme için doğrulama bölmesini nasıl yaparsınız?

P1 planınıza hedef birimi, büyüklük dağılımını, bilinen/yeni küme hedefini ve parametre belirsizliğinin nasıl ele alınacağını ekleyin.

- Kümelenmiş veriyi bağımsız sayıp okul düzeyindeki etkileri aşırı kesin raporlamak.

- Ham bir yordayıcıyla tek eğim kestirip okul içi ve okullar arası ilişkileri karıştırmak.

- Rastgele eğim varyansını naif ki-kare veya Wald z ile sınamak.

- Sabit etkileri farklı modelleri REML olabilirlikleriyle karşılaştırmak.

- Yazılımın “yakınsadı” iletisini denetlemeden kabul etmek; sınırdaki varyansları gözden kaçırmak.

# Alıştırmalar

## Kolay Düzey

1.  Sınıf içi korelasyon (ICC) neyi gösterir?

2.  Rastgele kesişim ile rastgele eğim arasındaki fark nedir?

3.  Grup ortalamasına merkezleme ne demektir?

4.  Tasarım etkisi neden küme büyüklüğüne bağlıdır?

5.  REML ne zaman, ML ne zaman kullanılır?

## Orta Düzey

1.  ICC=.05, küme büyüklüğü 30’dur. Tasarım etkisini hesaplayıp çok düzeyli modelin gerekliliğini tartışın.

2.  Grup içi eğim 1.5, gruplar arası eğim 1.5 bulunmuştur. Bağlamsal etki nedir, ne anlama gelir?

3.  Bir yordayıcı eklendiğinde $\tau_{00}$ artmıştır. Bu nasıl mümkündür?

4.  12 okuldan toplanan veride okul düzeyindeki bir değişkenin etkisi sınanacaktır. Hangi sorunlar beklenir?

5.  Rastgele eğim varyansı sıfıra çok yakın ve yazılım uyarı veriyor. Ne yaparsınız?

## İleri Düzey

1.  Ekolojik ve atomistik yanılgıyı HSB örneği üzerinden açıklayın.

2.  Rastgele etkilerin yordayıcılarla ilişkili olması hangi yanlılığa yol açar? Grup ortalamasına merkezleme bu sorunu nasıl ele alır?

3.  Düzeyler arası etkileşimde düzey 1 yordayıcısını genel ortalamaya merkezlemek yorumu nasıl değiştirir?

4.  Küme-sağlam standart hatalı OLS ile çok düzeyli model ne zaman tercih edilir?

5.  Güçlü bir çok düzeyli model raporunda bulunması gereken en az beş unsuru yazın.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b16/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Öğrenci ağırlıklı veya okul ağırlıklı hedefi seçin; ortalama ve yorum biriminin değişimini gösterin.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 16,
    "gorev": 'Öğrenci ağırlıklı veya okul ağırlıklı hedefi seçin; ortalama ve yorum biriminin değişimini gösterin.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":16,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b16-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b16-cok-duzeyli-modeller',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)